# 🧪 Baseline Model: Quality Score Prediction

**Project:** Gemma 4 - Good AI for Humanity, Not Just Performance  
**Notebook:** Baseline regression model for predicting AI response quality  
**Author:** Data Science Team  
**Last Updated:** 2024-02-20

## 📋 Table of Contents
1. [Introduction](#introduction)
2. [Data Loading & Preprocessing](#data-loading)
3. [Feature Engineering](#feature-engineering)
4. [Train/Validation Split](#data-split)
5. [Baseline Models](#baseline-models)
6. [Model Comparison](#model-comparison)
7. [Error Analysis](#error-analysis)
8. [Conclusions](#conclusions)

---
## 1. Introduction <a name="introduction"></a>

This notebook establishes baseline models for predicting AI response quality scores.  
**Target:** `composite_score` (weighted combination of humanity, performance, helpfulness)  
**Goal:** Establish performance benchmarks before advanced experiments.

In [ ]:
# ============================================
# Cell 1: Imports
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries loaded!")

---
## 2. Data Loading & Preprocessing <a name="data-loading"></a>

In [ ]:
# ============================================
# Cell 2: Load Data
# ============================================
DATA_DIR = '../data/raw/'

train_df = pd.read_csv(f'{DATA_DIR}train.csv')
labels_df = pd.read_csv(f'{DATA_DIR}labels.csv')

# Merge datasets
df = train_df.merge(
    labels_df[['id', 'safety_score', 'fairness_score', 'clarity_score', 'annotator_id']],
    on='id', how='left'
)

# Compute composite target
df['composite_score'] = (
    df['humanity_score'] * 0.35 +
    df['performance_score'] * 0.25 +
    df['helpfulness_score'] * 0.40
)

print(f"Merged dataset: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")

In [ ]:
# ============================================
# Cell 3: Clean Data
# ============================================
# Fill missing annotation scores with median
annotation_cols = ['safety_score', 'fairness_score', 'clarity_score']
for col in annotation_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Parse timestamps
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Drop rows with missing essential data
df = df.dropna(subset=['prompt', 'response'])

print(f"Clean dataset: {df.shape}")

---
## 3. Feature Engineering <a name="feature-engineering"></a>

In [ ]:
# ============================================
# Cell 4: Extract Features
# ============================================
def engineer_features(df):
    """Create features from text and metadata."""
    feat = df.copy()
    
    # Text length features
    feat['prompt_length'] = feat['prompt'].str.len()
    feat['response_length'] = feat['response'].str.len()
    feat['prompt_word_count'] = feat['prompt'].str.split().str.len()
    feat['response_word_count'] = feat['response'].str.split().str.len()
    feat['length_ratio'] = feat['response_word_count'] / feat['prompt_word_count']
    
    # Structural features
    feat['sentence_count'] = feat['response'].apply(lambda x: len(re.split(r'[.!?]+', str(x))))
    feat['avg_sentence_length'] = feat['response_word_count'] / feat['sentence_count']
    feat['has_list'] = feat['response'].apply(lambda x: int(bool(re.search(r'(\d\)|-\s|\*)', str(x)))))
    feat['has_number'] = feat['response'].apply(lambda x: int(bool(re.search(r'\d', str(x)))))
    feat['paragraph_count'] = feat['response'].apply(lambda x: str(x).count('\n\n') + 1)
    
    # Lexical diversity
    feat['unique_word_count'] = feat['response'].apply(
        lambda x: len(set(str(x).lower().split()))
    )
    feat['lexical_diversity'] = feat['unique_word_count'] / feat['response_word_count']
    
    # Question/exclamation marks
    feat['question_marks'] = feat['response'].str.count('\?')
    feat['exclamation_marks'] = feat['response'].str.count('!')
    
    # Categorical encoding
    le = LabelEncoder()
    feat['category_encoded'] = le.fit_transform(feat['category'])
    feat['difficulty_encoded'] = le.fit_transform(feat['difficulty'])
    feat['language_encoded'] = le.fit_transform(feat['language'])
    
    return feat

df = engineer_features(df)
print(f"Feature-engineered dataset: {df.shape}")
print(f"\nNew features added: {len([c for c in df.columns if c not in train_df.columns])}")

---
## 4. Train/Validation Split <a name="data-split"></a>

In [ ]:
# ============================================
# Cell 5: Define Features and Target
# ============================================
FEATURE_COLS = [
    'prompt_length', 'response_length', 'prompt_word_count', 'response_word_count',
    'length_ratio', 'sentence_count', 'avg_sentence_length', 'has_list', 'has_number',
    'paragraph_count', 'unique_word_count', 'lexical_diversity',
    'question_marks', 'exclamation_marks',
    'category_encoded', 'difficulty_encoded', 'language_encoded',
    'hour', 'day_of_week'
]

# Add annotation scores if available (for training only - not for prediction)
available_annotation_cols = [c for c in annotation_cols if c in df.columns]
train_feature_cols = FEATURE_COLS + available_annotation_cols

X = df[train_feature_cols]
y = df['composite_score']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=pd.qcut(y, q=5, duplicates='drop')
)

print(f"Training set:   {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"\nTarget distribution:")
print(f"  Train mean: {y_train.mean():.3f}, std: {y_train.std():.3f}")
print(f"  Val   mean: {y_val.mean():.3f}, std: {y_val.std():.3f}")

---
## 5. Baseline Models <a name="baseline-models"></a>

In [ ]:
# ============================================
# Cell 6: Scale Features
# ============================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Features scaled!")

In [ ]:
# ============================================
# Cell 7: Train Multiple Baseline Models
# ============================================
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='neg_root_mean_squared_error')
    
    results[name] = {
        'RMSE': rmse, 'MAE': mae, 'R2': r2,
        'CV RMSE Mean': -cv_scores.mean(), 'CV RMSE Std': cv_scores.std()
    }
    
    print(f"{name:25s} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f} | CV-RMSE: {-cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

---
## 6. Model Comparison <a name="model-comparison"></a>

In [ ]:
# ============================================
# Cell 8: Model Comparison Visualization
# ============================================
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE
axes[0].barh(results_df.index, results_df['RMSE'], color='#e74c3c', alpha=0.8)
axes[0].set_title('RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('RMSE')
for i, v in enumerate(results_df['RMSE']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# MAE
axes[1].barh(results_df.index, results_df['MAE'], color='#3498db', alpha=0.8)
axes[1].set_title('MAE (Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('MAE')
for i, v in enumerate(results_df['MAE']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# R2
axes[2].barh(results_df.index, results_df['R2'], color='#2ecc71', alpha=0.8)
axes[2].set_title('R2 Score (Higher is Better)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('R2')
for i, v in enumerate(results_df['R2']):
    axes[2].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Baseline Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/baseline_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# Cell 9: Best Model Predictions vs Actual
# ============================================
best_model_name = results_df['R2'].idxmax()
print(f"Best baseline model: {best_model_name}")

best_model = models[best_model_name]
y_pred_best = best_model.predict(X_val_scaled)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_val, y_pred_best, alpha=0.7, color='#3498db', edgecolors='gray', s=80)
ax.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Composite Score', fontsize=12)
ax.set_ylabel('Predicted Composite Score', fontsize=12)
ax.set_title(f'{best_model_name}: Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig('../data/processed/baseline_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Error Analysis <a name="error-analysis"></a>

In [ ]:
# ============================================
# Cell 10: Error Analysis
# ============================================
errors = pd.DataFrame({
    'actual': y_val,
    'predicted': y_pred_best,
    'error': y_val - y_pred_best,
    'abs_error': np.abs(y_val - y_pred_best)
})

errors['category'] = df.loc[y_val.index, 'category'].values
errors['difficulty'] = df.loc[y_val.index, 'difficulty'].values

print("Error Statistics:")
print(errors[['error', 'abs_error']].describe())

# Error by category
error_by_cat = errors.groupby('category')['abs_error'].agg(['mean', 'std', 'count']).round(3)
print("\nAbsolute Error by Category:")
print(error_by_cat.sort_values('mean', ascending=False))

In [ ]:
# ============================================
# Cell 11: Residual Plot
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual scatter
axes[0].scatter(y_pred_best, errors['error'], alpha=0.7, color='#e74c3c', edgecolors='gray', s=60)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_xlabel('Predicted Score')
axes[0].set_ylabel('Residual (Actual - Predicted)')
axes[0].set_title('Residual Plot', fontsize=12, fontweight='bold')

# Error distribution
axes[1].hist(errors['error'], bins=15, color='#9b59b6', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Error Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/baseline_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Conclusions <a name="conclusions"></a>

In [ ]:
# ============================================
# Cell 12: Summary & Export
# ============================================
print("=" * 60)
print("BASELINE MODEL - SUMMARY")
print("=" * 60)
print(f"\nBest Model: {best_model_name}")
print(f"  RMSE: {results_df.loc[best_model_name, 'RMSE']:.4f}")
print(f"  MAE:  {results_df.loc[best_model_name, 'MAE']:.4f}")
print(f"  R2:   {results_df.loc[best_model_name, 'R2']:.4f}")
print(f"\nAll Model Results:")
print(results_df.round(4).to_string())

# Save results
results_df.to_csv('../data/processed/baseline_results.csv')
errors.to_csv('../data/processed/baseline_errors.csv', index=False)

print("\nResults saved to ../data/processed/")

---
## ✅ Baseline Complete

**Next Steps:**
- Try TF-IDF features in `tfidf_experiment.ipynb`
- Test Gemma prompt strategies in `gemma_prompt_test.ipynb`
- Combine best approaches in `gemma_4_good_final.ipynb`